# Wordle with `games`
This notebook explores the Wordle implementation in the `games` Python library.

We will use the library to:

1. Load the Wordle vocabulary.
2. Generate and interpret Wordle feedback.
3. Represent a Wordle position as a `WordleState`.
4. Watch the candidate solution space shrink after each guess.
5. Construct and step through a `WordleGame`.
6. Build a simple notebook-based player.
7. Visualize the game as it develops.

The important idea is that Wordle is not just a word-guessing game. At every point in the game, the player has a **position**: a set of information that constrains which solutions remain possible.

The `games` library represents that position explicitly and lets actions, rules, and state interact through a common simulation framework.

## Import the Wordle library

The Wordle implementation is deliberately split into several pieces:

- `WordleGuess` represents an action.
- `feedback()` produces the observable result of a guess.
- `interpret_feedback()` converts that result into constraints.
- `WordleState` represents the current position.
- `WordleGame` combines the state, rules, and player into a simulation.

In [ ]:
if "google.colab" in str(get_ipython()):
  !pip install git+https://github.com/DiogenesAnalytics/games

In [ ]:
from games.catalog.simulation.wordle.action import WordleGuess
from games.catalog.simulation.wordle.feedback import FeedbackTile
from games.catalog.simulation.wordle.feedback import feedback
from games.catalog.simulation.wordle.feedback import interpret_feedback
from games.catalog.simulation.wordle.game import WordleGame
from games.catalog.simulation.wordle.state import WordleState
from games.catalog.simulation.wordle.words import load_answers
from games.catalog.simulation.wordle.words import load_words
from games.visualization.wordle.terminal import TerminalWordleRenderer

## Load the Wordle vocabulary

The answer list contains the words that can actually be selected as hidden solutions.

The complete word list contains every word that can legally be submitted as a guess.

This distinction matters: a player may be able to guess a word that can never itself be the hidden answer.

In [ ]:
answers = load_answers()
words = load_words()

print(f"Answer words: {len(answers):,}")
print(f"Legal guesses: {len(words):,}")

In [ ]:
print("Sample answers:")
print(sorted(answers)[:20])

print()
print("Sample legal guesses:")
print(sorted(words)[:20])

## A Wordle guess is an Action
A guess is represented by `WordleGuess`.

The action itself does not decide whether the guess is legal. That responsibility belongs to a rule.

In [ ]:
guess = WordleGuess("crane")

print(guess)
print(guess.word)
print(guess.is_valid)
print(guess.is_resolved)

Notice that the guess is normalized to uppercase.

It also starts unresolved. Creating an action does not automatically make it legal or executable.

The simulation will later pass the action through its rules.

## Generate Wordle feedback
The central observation in Wordle is the feedback produced when a guess is compared with the hidden target.

The `feedback()` function returns a structured tuple of `FeedbackTile` values.

In [ ]:
target = "CRANE"

guesses = [
    "SLATE",
    "CRATE",
    "CRONE",
    "CRANE",
]

for guess in guesses:
    result = feedback(target, guess)
    print(f"{guess}: {result}")

The raw feedback is useful when examining the implementation, but it is not how a player normally experiences the game.

For that, we use the game's renderer.

In [ ]:
renderer = TerminalWordleRenderer()

state = WordleState(
    solutions=answers,
    available_guesses=words,
    target=target,
)

## Render a Wordle position
The `TerminalWordleRenderer` converts the structured `WordleState` into the familiar Wordle board.

This is the same renderer used by the command-line interface.

In [ ]:
print(renderer.render(state))

At the beginning of the game, the board is empty.

The state already contains the complete solution universe and the hidden target, but no guesses or observations have yet been recorded.

## Make the first guess
Let's make `SLATE` as the first guess.

The feedback is calculated from the target and then recorded in the state.

In [ ]:
guess = "SLATE"
result = feedback(state.target, guess)

state.update((guess, result))

print(renderer.render(state))

The rendered board now shows the information produced by the first guess.

The target and complete solution universe remain unchanged.

What changed is the candidate set: words inconsistent with the observation were eliminated.

In [ ]:
print(f"Candidates remaining: {len(state.candidates):,}")

## Inspect the remaining candidates
The candidate set is the current hypothesis space for the hidden solution.

In [ ]:
print(sorted(state.candidates))

At this point, the player does not know the target directly.

The player knows that the target must belong to the remaining candidate set.

## Duplicate letters
Duplicate letters make Wordle's feedback rules particularly interesting.

Consider the target `LOOSE`.

We can inspect several guesses against it and render each resulting position.

In [ ]:
duplicate_guesses = [
    "MOOSE",
    "BOOST",
    "SLOOP",
    "OOOZE",
]

for guess in duplicate_guesses:
    duplicate_state = WordleState(
        solutions=answers,
        available_guesses=words,
        target="LOOSE",
    )

    result = feedback(duplicate_state.target, guess)
    duplicate_state.update((guess, result))

    print(renderer.render(duplicate_state))
    print()

The renderer lets us see the actual Wordle result rather than exposing the internal feedback representation.

Underneath that rendering, the feedback algorithm is accounting for how many copies of each letter actually occur in the target.

## Interpret feedback as constraints
Raw feedback is useful to a human player, but a solver needs something more explicit.

`interpret_feedback()` converts the five feedback tiles into constraints describing what the hidden word can and cannot contain.

In [ ]:
guess = "SLATE"
result = feedback("CRANE", guess)

information = interpret_feedback(guess, result)

print("Required positions:")
print(information.required)

print()
print("Excluded positions:")
print(information.excluded)

print()
print("Letter counts:")
print(information.counts)

The resulting information distinguishes several kinds of constraints:

- letters known to occupy specific positions;
- letters excluded from specific positions;
- minimum letter counts;
- maximum letter counts.

These constraints are what allow the state to reduce the solution space.

## Create a Wordle state
A `WordleState` represents a complete Wordle position.

It contains the fixed solution universe, the hidden target, the current candidate set, previous guesses, and their feedback.

In [ ]:
state = WordleState(
    solutions=answers,
    available_guesses=words,
    target="CRANE",
)

print(f"Solutions: {len(state.solutions):,}")
print(f"Candidates: {len(state.candidates):,}")
print(f"Guesses: {state.guesses}")

Initially, every possible answer is a candidate.

No observations have been made yet, so nothing has been eliminated.

## Watch the position evolve
Let's reset the state and make several guesses in sequence.

After every guess, we render the complete board and report the remaining candidate count.

In [ ]:
guesses = [
    "SLATE",
    "CHOSE",
    "CRONE",
    "CRANE",
]

state.reset()

for guess in guesses:
    result = feedback(state.target, guess)
    state.update((guess, result))

    print(renderer.render(state))

    if state.target in state.guesses:
        print("Solved!")
    else:
        print(f"Possible solutions: {len(state.candidates):,}")

    print()

The game can therefore be viewed as a sequence of changing positions.

A guess produces an observation, and the observation restricts the next position.

$$
P_t
\longrightarrow
a_t
\longrightarrow
o_t
\longrightarrow
P_{t+1}
$$

For Wordle, the candidate set is one particularly useful part of that position.

## Measure the shrinking hypothesis space
The candidate count gives us a simple quantitative measure of how much the game position has changed.

In [ ]:
state.reset()

candidate_counts = [len(state.candidates)]

for guess in guesses:
    result = feedback(state.target, guess)
    state.update((guess, result))
    candidate_counts.append(len(state.candidates))

for turn, count in enumerate(candidate_counts):
    print(f"After turn {turn}: {count:,} candidates")

We can visualize that change.

In [ ]:
import matplotlib.pyplot as plt

turns = range(len(candidate_counts))

fig, ax = plt.subplots(figsize=(9, 4))

fig.patch.set_facecolor("#121212")
ax.set_facecolor("#121212")

ax.plot(
    turns,
    candidate_counts,
    marker="o",
    linewidth=2,
)

ax.set_xticks(list(turns))
ax.set_xlabel("Turn", color="white")
ax.set_ylabel("Candidate solutions", color="white")
ax.set_title("Wordle hypothesis space", color="white")

ax.tick_params(colors="white")
ax.grid(True, alpha=0.2)

for spine in ax.spines.values():
    spine.set_color("#444444")

plt.tight_layout()
plt.show()

## The complete game abstraction
So far we have manipulated `WordleState` directly.

The library also provides `WordleGame`, which assembles the state, rules, and player into a simulation.

Conceptually:

```text
Actor
  │
  │ decide
  ▼
Action
  │
  │ validate / resolve
  ▼
Rule
  │
  │ execute
  ▼
State
```

The game therefore separates:

+ who proposes an action
+ whether that action is legal
+ how that action changes the state

## Create a notebook player
The normal interactive player obtains guesses from `input()`.

For a notebook, we can instead provide a sequence of guesses programmatically.

The player still implements the same `Player` interface used by the game.

In [ ]:
from typing import Iterator
from typing import List

from games.primitive.action.base import Action
from games.primitive.actor.base import Player
from games.primitive.state.base import State


class NotebookWordlePlayer(Player):
    """A Wordle player whose guesses are supplied programmatically."""

    def __init__(self, guesses: List[str]) -> None:
        """Initialize the player with a sequence of guesses."""
        self._guesses: Iterator[str] = iter(guesses)

    def decide(self, state: State) -> Action:
        """Return the next Wordle guess."""
        return WordleGuess(next(self._guesses))

The game does not need to know whether the player is human, scripted, random, or algorithmic.

It only needs an actor capable of deciding on an action from the current state.

## Play a complete game
Now we can let `WordleGame` run the complete actor → action → rule → state cycle.

In [ ]:
player = NotebookWordlePlayer(
    [
        "SLATE",
        "CHOSE",
        "CRONE",
        "CRANE",
    ]
)

game = WordleGame(
    solutions=answers,
    available_guesses=words,
    target="CRANE",
    player=player,
)

In [ ]:
while not game.is_done():
    game.step()

    print(renderer.render(game.state))

    if game.state.target in game.state.guesses:
        print("Solved!")
    else:
        print(f"Possible solutions: {len(game.state.candidates):,}")

    print()

Here we are no longer manually calculating feedback or updating the state.

`WordleGame.step()` runs the complete simulation cycle.

The player proposes a `WordleGuess`, the rules validate and resolve it, and the resulting action updates the `WordleState`.

## Inspect the finished game
The board is a rendering of the state, but the state itself remains available for inspection.

In [ ]:
print("Target:")
print(game.state.target)

print()
print("Guesses:")
print(game.state.guesses)

print()
print("Terminal:")
print(game.state.is_terminal)

print()
print("Solved:")
print(game.state.target in game.state.guesses)

print()
print("Final board:")
print(renderer.render(game.state))

## Render the game at any point
Because rendering is independent of the game mechanics, the same renderer can be used whenever we want to inspect the current position.

For example, we can reset the game state and render the board again.

In [ ]:
game.state.reset()

print(renderer.render(game.state))

Resetting the state removes the observations and restores the original candidate set.

The hidden target remains the same.

In [ ]:
print(f"Candidates after reset: {len(game.state.candidates):,}")

## The Wordle position
The most interesting object in this example is not the board.

It is the changing set of possible solutions.

Let $S$ be the complete solution universe.

Initially:

$$
C_0 = S
$$

After a guess produces an observation $o_t$, the next candidate set contains only words consistent with that observation:

$$
C_{t+1}
=
\{w \in C_t : w \text{ is consistent with } o_t\}
$$

Thus each observation restricts the future possibilities.

The Wordle game can be viewed as a deterministic sequence of positions:

$$
C_0
\rightarrow
C_1
\rightarrow
C_2
\rightarrow
\cdots
$$

The player chooses an action from one position, receives an observation, and arrives at another position.